# Atividade NLP

## Notebook para a disciplina de Redes Neurais do Mestrado em Inteligência Artificial

> Aluno: André Vitória

Este notebook foi construido com base nos trabalhos de [Eesuck](https://www.kaggle.com/code/eesuck/bert-natural-language).


In [ ]:
#!pip install transformers
import tensorflow as tf
import pandas
from transformers import BertTokenizer, TFBertForSequenceClassification

model_name = 'bert-base-uncased'

In [ ]:
from sklearn.model_selection import train_test_split

df = pandas.read_csv("/kaggle/input/nlp-getting-started/train.csv")

train_data, val_data = train_test_split(df, test_size=0.2, random_state=42)

test_data = pandas.read_csv("/kaggle/input/nlp-getting-started/test.csv")



In [ ]:
train_data.head()

In [ ]:
val_data.head()

In [ ]:
test_data.head()

In [ ]:
train_labels = train_data.pop("target")
val_labels = val_data.pop("target")

In [ ]:
train_data.fillna("0", inplace=True)
val_data.fillna("0", inplace=True)
test_data.fillna("0", inplace=True)

In [ ]:
train_data.head(10)

In [ ]:
test_data.head(10)

In [ ]:
def my_tokenizer(df, model_name):
    tokenizer = BertTokenizer.from_pretrained(model_name)
    return tokenizer((df["text"] + " [ " + df["keyword"] + " ] " + df["location"]).to_list(), truncation=True, padding=True)


train_tokinized = my_tokenizer(train_data, model_name)
val_tokinized = my_tokenizer(val_data, model_name)
test_tokinized = my_tokenizer(test_data, model_name)

In [ ]:
dict(train_tokinized).keys(), dict(val_tokinized).keys(), dict(test_tokinized).keys()

In [ ]:
train_dataset = tf.data.Dataset.from_tensor_slices((
    dict(train_tokinized),
    train_labels
))

val_dataset = tf.data.Dataset.from_tensor_slices((
    dict(val_tokinized),
    val_labels
))

test_dataset = tf.data.Dataset.from_tensor_slices((
    dict(test_tokinized),
))

In [ ]:
train_dataset, val_dataset, test_dataset

In [ ]:
model = TFBertForSequenceClassification.from_pretrained(model_name)

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=3e-5),
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# Train the model
checkpointer = ModelCheckpoint(filepath='../working/my_model/', 
                               verbose=1, save_best_only=True)
early_stop = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=10)

model.fit(train_dataset.shuffle(1000).batch(16),
          epochs=50,
          validation_data=val_dataset.batch(16),
          callbacks=[
              checkpointer,
              early_stop
          ],
          batch_size=16)

In [ ]:
predictions = model.predict(test_dataset.batch(16))

In [ ]:
result = pandas.read_csv("/kaggle/input/nlp-getting-started/sample_submission.csv")
result["target"] = tf.argmax(predictions.logits, axis=1).numpy()

result.to_csv("submission.csv", index=False)